## Operator Benchmarks

Each row checks forward and backward correctness against `TorchOpsBackend`, then reports forward, backward-only, and full forward+backward p50 latency, p95 latency, peak CUDA memory delta, and speedup versus torch.


In [ ]:
from pathlib import Path
import sys

import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "minitrain").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "tests"))

from minitrain.model.ops import get_ops_backend
from operator_nsight import nsight_kernel, register_nsight_kernel
from operator_bench_utils import (
    BenchCase,
    benchmark_architecture_dir,
    bench_sweep,
    collect_triton_best_configs,
    display_triton_config_summary,
    plot_kernel_grid,
    save_benchmark_results,
    to_summary_dataframe,
)

torch.manual_seed(0)
assert torch.cuda.is_available(), "operator benchmarks require CUDA"

DEVICE = torch.device("cuda")
DTYPE = torch.float16
PROVIDERS = ("torch", "triton", "cuda")
WARMUP_MS = 10
REP_MS = 50
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
BENCHMARK_CACHE_DIR = PROJECT_ROOT / "tests" / "benchmark_results"
GPU_BENCHMARK_DIR = benchmark_architecture_dir(BENCHMARK_CACHE_DIR)
NSIGHT_REPORT_DIR = GPU_BENCHMARK_DIR / "nsight"
METRICS = (
    "fwd_p50_ms",
    "fwd_p95_ms",
    "fwd_peak_mem_mb",
    "fwd_speedup",
    "bwd_p50_ms",
    "bwd_p95_ms",
    "bwd_peak_mem_mb",
    "bwd_speedup",
    "full_p50_ms",
    "full_p95_ms",
    "full_peak_mem_mb",
    "full_speedup",
)

print(torch.__version__)
print(torch.cuda.get_device_name(0))


Each kernel section now provides `make_case(size)` and `forward(provider, tensors)`. The runner owns tensor cloning, forward/backward correctness, timing, memory measurement, and cleanup. Cleanup clears the torch-created tensor dictionary before releasing the CUDA cache.

Every completed dataset is cached in `tests/benchmark_results/<sm-arch>-<gpu-name>/<operator>/<UTC timestamp>.json`. Repeated runs are retained independently. The strict JSON cache retains raw correctness, latency, memory, speedup, Triton configuration, unsupported-branch, and error fields for later README generation.

Generate and cache a small Nsight Compute report with one line, for example `profile_and_cache_kernel("rmsnorm")` or `profile_and_cache_kernel("attention", mode="bwd")`. `size` uses the same logical size as the corresponding benchmark when explicitly provided. The default `LaunchStats` section reports launch geometry, registers, and shared memory; pass `set_name="basic"` for a broader, slower analysis. Autotuned kernels are tuned once before ncu starts, then the worker is pinned to the winning configuration.

In [ ]:
def register_kernel(
    kernel,
    sizes,
    size_label,
    make_case,
    forward,
    *,
    nsight_size=None,
    nsight_autotune_kernels=None,
):
    sizes = tuple(sizes)
    register_nsight_kernel(
        kernel,
        make_case,
        forward,
        default_size=min(sizes) if nsight_size is None else nsight_size,
        autotune_kernels=nsight_autotune_kernels,
    )
    return sizes


def cache_benchmark_rows(benchmark, rows, **metadata):
    cache_path = save_benchmark_results(
        rows,
        benchmark=benchmark,
        cache_root=BENCHMARK_CACHE_DIR,
        metadata={
            "warmup_ms": WARMUP_MS,
            "rep_ms": REP_MS,
            **metadata,
        },
    )
    print(f"raw benchmark data: {cache_path}")
    return cache_path


def profile_and_cache_kernel(name, **kwargs):
    kwargs.setdefault("report_dir", NSIGHT_REPORT_DIR)
    result = nsight_kernel(name, **kwargs)
    cache_benchmark_rows(
        f"nsight_{name}_{result['mode']}_{result['size']}",
        [result],
        dataset_type="nsight",
    )
    return result


def run_kernel(
    kernel,
    sizes,
    size_label,
    make_case,
    forward,
    *,
    providers=None,
    autotune_kernels=None,
    nsight_size=None,
    nsight_autotune_kernels=None,
):
    sizes = register_kernel(
        kernel,
        sizes,
        size_label,
        make_case,
        forward,
        nsight_size=nsight_size,
        nsight_autotune_kernels=nsight_autotune_kernels,
    )
    providers = PROVIDERS if providers is None else tuple(providers)
    rows = bench_sweep(
        kernel=kernel,
        providers=providers,
        sizes=sizes,
        size_label=size_label,
        make_case=make_case,
        forward=forward,
        warmup_ms=WARMUP_MS,
        rep_ms=REP_MS,
        autotune_kernels=autotune_kernels,
    )
    cache_benchmark_rows(
        kernel,
        rows,
        providers=providers,
        size_label=size_label,
    )
    display(to_summary_dataframe(rows))
    plot_kernel_grid(rows, metrics=METRICS, save_path=FIG_DIR / f"{kernel}_summary.png")
    return rows


## RMSNorm

Sweep parameter: `rows`.

X-axis: activation tensor elements, `rows * hidden`.

In [ ]:
PROVIDERS = ("torch", "triton")
def make_rmsnorm_case(size):
    hidden = 1024
    rows = size // hidden
    return BenchCase(
        tensors={
            "x": torch.randn(rows, hidden, device=DEVICE, dtype=DTYPE),
            "weight": torch.ones(hidden, device=DEVICE, dtype=DTYPE),
        },
        grad_names=("x", "weight"),
    )


def rmsnorm_forward(provider, tensors):
    backend = get_ops_backend(provider)
    return backend.rmsnorm(tensors["x"], tensors["weight"], 1e-5)


rmsnorm_sizes = [1024 * rows for rows in (64, 128, 256, 512, 1024)]
rmsnorm_rows = run_kernel(
    "rmsnorm",
    rmsnorm_sizes,
    "rows * hidden elements",
    make_rmsnorm_case,
    rmsnorm_forward,
)

## RoPE

Sweep parameter: `seq`.

X-axis: Q+K tensor elements, `2 * batch * heads * seq * head_dim`.

In [ ]:
PROVIDERS = ("torch", "triton")
def rope_cache(seq, dim):
    inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2, device=DEVICE).float() / dim))
    freqs = torch.outer(torch.arange(seq, device=DEVICE).float(), inv_freq)
    emb = torch.cat((freqs, freqs), dim=-1)
    return emb.cos().to(dtype=DTYPE), emb.sin().to(dtype=DTYPE)


def make_rope_case(size):
    batch, heads, head_dim = 1, 8, 64
    seq = size // (2 * batch * heads * head_dim)
    cos, sin = rope_cache(seq, head_dim)
    return BenchCase(
        tensors={
            "q": torch.randn(batch, heads, seq, head_dim, device=DEVICE, dtype=DTYPE),
            "k": torch.randn(batch, heads, seq, head_dim, device=DEVICE, dtype=DTYPE),
            "cos": cos,
            "sin": sin,
        },
        grad_names=("q", "k"),
    )


def rope_forward(provider, tensors):
    backend = get_ops_backend(provider)
    return backend.rope(tensors["q"], tensors["k"], tensors["cos"], tensors["sin"])


rope_sizes = [2 * 1 * 8 * seq * 64 for seq in (128, 512, 1024, 2048, 4096)]
rope_rows = run_kernel("rope", rope_sizes, "Q + K elements", make_rope_case, rope_forward)


## SwiGLU

Sweep parameter: `rows`.

X-axis: input tensor elements, `2 * rows * intermediate` for gate and up.

In [ ]:
PROVIDERS = ("torch", "triton")
def make_swiglu_case(size):
    intermediate = 1024
    rows = size // (2 * intermediate)
    return BenchCase(
        tensors={
            "gate": torch.randn(rows, intermediate, device=DEVICE, dtype=DTYPE),
            "up": torch.randn(rows, intermediate, device=DEVICE, dtype=DTYPE),
        },
        grad_names=("gate", "up"),
    )


def swiglu_forward(provider, tensors):
    backend = get_ops_backend(provider)
    return backend.swiglu(tensors["gate"], tensors["up"])


swiglu_sizes = [2 * rows * 1024 for rows in (64, 128, 256, 512, 1024)]
swiglu_rows = run_kernel(
    "swiglu",
    swiglu_sizes,
    "gate + up elements",
    make_swiglu_case,
    swiglu_forward,
)


## CrossEntropy

Sweep parameter: `vocab` at a fixed 256 tokens.

X-axis: logits tensor elements, `tokens * vocab`.

In [ ]:
PROVIDERS = ("torch", "triton")
def make_cross_entropy_case(size):
    tokens = 1024
    vocab = size // tokens
    return BenchCase(
        tensors={
            "logits": torch.randn(tokens, vocab, device=DEVICE, dtype=DTYPE),
            "targets": torch.randint(vocab, (tokens,), device=DEVICE),
        },
        grad_names=("logits",),
    )


def cross_entropy_forward(provider, tensors):
    backend = get_ops_backend(provider)
    # Match the trainer's mixed-precision contract: CE returns an fp32 scalar.
    with torch.autocast("cuda", dtype=DTYPE):
        return backend.cross_entropy(tensors["logits"], tensors["targets"])


cross_entropy_sizes = [1024 * vocab for vocab in (1024, 2048, 4096, 8192, 16384, 32768)]
cross_entropy_rows = run_kernel(
    "cross_entropy",
    cross_entropy_sizes,
    "1024 tokens * vocab logits elements",
    make_cross_entropy_case,
    cross_entropy_forward,
)

## FusedLinearCrossEntropy

Sweep parameter: `vocab`.

X-axis: logical logits elements, `tokens * vocab`. This is the tensor a fused implementation tries not to materialize.

In [ ]:
def make_fused_linear_ce_case(size):
    tokens, hidden = 256, 512
    vocab = size // tokens
    return BenchCase(
        tensors={
            "x": torch.randn(tokens, hidden, device=DEVICE, dtype=DTYPE),
            "weight": torch.randn(vocab, hidden, device=DEVICE, dtype=DTYPE),
            "targets": torch.randint(vocab, (tokens,), device=DEVICE),
        },
        grad_names=("x", "weight"),
    )


def fused_linear_ce_forward(provider, tensors):
    backend = get_ops_backend(provider)
    # Exercise the same autocast path used by MiniTransformer training.
    with torch.autocast("cuda", dtype=DTYPE):
        return backend.fused_linear_cross_entropy(
            tensors["x"], tensors["weight"], tensors["targets"]
        )


fused_linear_ce_sizes = [256 * vocab for vocab in (1024, 2048, 4096, 8192, 16384, 32768)]
fused_linear_ce_rows = run_kernel(
    "fused_linear_cross_entropy",
    fused_linear_ce_sizes,
    "256 tokens * vocab logical logits elements",
    make_fused_linear_ce_case,
    fused_linear_ce_forward,
)


## FlashAttention


### Shared configuration and helpers

One QKV case factory and one provider-checked forward factory are reused by the no-dropout sweep, dropout sweep, and specialization matrix.


In [ ]:
from functools import partial
from itertools import product
import math

import pandas as pd
import torch.nn.functional as F
from torch.profiler import ProfilerActivity

from minitrain.kernels.cuda_ext.build import get_build_config as get_cuda_build_config
from minitrain.kernels.cuda_ext.flash_attention import (
    flash_attention_dropout_mask_for_testing as cuda_flash_attention_dropout_mask,
    is_flash_attention_supported as is_cuda_flash_attention_supported,
)
from minitrain.kernels.triton.flash_attention import (
    flash_attention_autotune_kernels,
    flash_attention_backward,
    flash_attention_dropout_mask,
    flash_attention_forward,
    is_flash_attention_supported as is_triton_flash_attention_supported,
)
from operator_bench_utils import benchmark_step, release_cache

ATTENTION_INPUT_NAMES = ("q", "k", "v")
ATTENTION_PROVIDERS = ("torch", "triton", "cuda")
ATTENTION_SIZE_LABEL = "Q + K + V elements"
ATTENTION_BENCHMARK_MODES = ("fwd", "bwd")

ATTENTION_BATCH_SIZE = 1
ATTENTION_NUM_HEADS = 4
ATTENTION_HEAD_DIM = 64
ATTENTION_SEQUENCE_LENGTHS = (256, 512, 1024, 2048)
ATTENTION_CAUSAL = True
ATTENTION_DROPOUT_RATE = 0.1

SDPA_ATEN_BACKENDS = (
    ("aten::_scaled_dot_product_flash_attention", "flash_attention"),
    ("aten::_scaled_dot_product_efficient_attention", "memory_efficient_attention"),
    ("aten::_scaled_dot_product_cudnn_attention", "cudnn_attention"),
    ("aten::_scaled_dot_product_attention_math", "math"),
)


def attention_element_count(
    sequence_length,
    *,
    batch_size=ATTENTION_BATCH_SIZE,
    num_heads=ATTENTION_NUM_HEADS,
    head_dim=ATTENTION_HEAD_DIM,
):
    return 3 * batch_size * num_heads * sequence_length * head_dim


def make_attention_case(
    size,
    *,
    batch_size=ATTENTION_BATCH_SIZE,
    num_heads=ATTENTION_NUM_HEADS,
    head_dim=ATTENTION_HEAD_DIM,
    dtype=DTYPE,
):
    elements_per_token = 3 * batch_size * num_heads * head_dim
    sequence_length, remainder = divmod(size, elements_per_token)
    if remainder:
        raise ValueError(f"size={size} is not divisible by {elements_per_token}")
    shape = (batch_size, num_heads, sequence_length, head_dim)
    return BenchCase(
        tensors={
            name: torch.randn(shape, device=DEVICE, dtype=dtype)
            for name in ATTENTION_INPUT_NAMES
        },
        grad_names=ATTENTION_INPUT_NAMES,
    )


def is_attention_provider_supported(provider, tensors, *, dropout_p):
    q, k, v = (tensors[name] for name in ATTENTION_INPUT_NAMES)
    if provider == "torch":
        return True
    if provider == "triton":
        return is_triton_flash_attention_supported(q, k, v, dropout_p=dropout_p)
    if provider == "cuda":
        return is_cuda_flash_attention_supported(q, k, v, dropout_p=dropout_p)
    raise ValueError(f"unknown attention provider: {provider}")


def attention_provider_support(tensors, *, dropout_p):
    return {
        provider: is_attention_provider_supported(
            provider, tensors, dropout_p=dropout_p
        )
        for provider in ATTENTION_PROVIDERS
    }


def make_attention_forward(*, is_causal, dropout_p):
    def forward(provider, tensors):
        if not is_attention_provider_supported(
            provider, tensors, dropout_p=dropout_p
        ):
            raise RuntimeError(
                f"{provider} does not support this attention specialization"
            )
        q, k, v = (tensors[name] for name in ATTENTION_INPUT_NAMES)
        return get_ops_backend(provider).attention(
            q,
            k,
            v,
            is_causal=is_causal,
            dropout_p=dropout_p,
        )

    return forward


def benchmark_attention_case(
    *,
    kernel,
    size,
    make_case,
    forward,
    metadata=None,
    correctness=None,
    supported=None,
):
    """Benchmark every provider for one shape and calculate Torch-relative speedups."""

    metadata = {} if metadata is None else metadata
    correctness = {} if correctness is None else correctness
    supported = {} if supported is None else supported
    torch_latency = {}
    rows = []

    try:
        for provider in ATTENTION_PROVIDERS:
            row = {
                "kernel": kernel,
                "provider": provider,
                "size": size,
                "size_label": ATTENTION_SIZE_LABEL,
                "status": "ok",
                **metadata,
                **correctness.get(provider, {}),
            }
            if not supported.get(provider, True):
                row["status"] = "unsupported"
                row.update({metric: math.nan for metric in METRICS})
                rows.append(row)
                continue

            try:
                for mode in ATTENTION_BENCHMARK_MODES:
                    row.update(
                        benchmark_step(
                            make_case,
                            size,
                            provider,
                            forward,
                            mode=mode,
                            warmup_ms=WARMUP_MS,
                            rep_ms=REP_MS,
                        )
                    )
                if provider == "triton":
                    row["triton_best_configs"] = collect_triton_best_configs(
                        flash_attention_autotune_kernels()
                    )
            except Exception as error:
                row.update(
                    status="unavailable",
                    error=f"{type(error).__name__}: {error}",
                )
                row.update({metric: math.nan for metric in METRICS})

            for mode in ATTENTION_BENCHMARK_MODES:
                latency = row.get(f"{mode}_p50_ms")
                if provider == "torch" and row["status"] == "ok":
                    torch_latency[mode] = latency
                    row[f"{mode}_speedup"] = 1.0
                else:
                    baseline = torch_latency.get(mode)
                    row[f"{mode}_speedup"] = (
                        baseline / latency if baseline and latency else math.nan
                    )
            rows.append(row)
        return rows
    finally:
        release_cache()


def detect_pytorch_sdpa_backend(q, k, v, *, dropout_p):
    activities = [ProfilerActivity.CPU, ProfilerActivity.CUDA]
    torch.cuda.synchronize()
    with torch.profiler.profile(activities=activities) as profiler:
        with torch.no_grad():
            F.scaled_dot_product_attention(
                q, k, v,
                is_causal=ATTENTION_CAUSAL,
                dropout_p=dropout_p,
            )
        torch.cuda.synchronize()

    event_keys = {event.key for event in profiler.key_averages()}
    for aten_name, backend_name in SDPA_ATEN_BACKENDS:
        if aten_name in event_keys:
            return backend_name
    events = sorted(
        key for key in event_keys
        if "scaled_dot_product" in key or "sdp" in key.lower()
    )
    return "unknown: " + ", ".join(events[:6])


def report_pytorch_sdpa_backends(sizes, *, dropout_p):
    print(
        "PyTorch SDPA backend selection "
        f"(causal={ATTENTION_CAUSAL}, dropout_p={dropout_p}):"
    )
    for size in sizes:
        case = make_attention_case(size)
        try:
            q, k, v = (case.tensors[name] for name in ATTENTION_INPUT_NAMES)
            backend = detect_pytorch_sdpa_backend(q, k, v, dropout_p=dropout_p)
            print(f"  shape={tuple(q.shape)} -> {backend}")
        finally:
            case.tensors.clear()
            release_cache()


### No-dropout sequence-length sweep

The sweep reports SDPA dispatch before comparing Torch, Triton, and CUDA.


In [ ]:
print("CUDA FlashAttention build:", get_cuda_build_config())

attention_sizes = tuple(
    attention_element_count(sequence_length)
    for sequence_length in ATTENTION_SEQUENCE_LENGTHS
)
attention_forward = make_attention_forward(
    is_causal=ATTENTION_CAUSAL,
    dropout_p=0.0,
)
report_pytorch_sdpa_backends(attention_sizes, dropout_p=0.0)
attention_rows = run_kernel(
    "attention",
    attention_sizes,
    ATTENTION_SIZE_LABEL,
    make_attention_case,
    attention_forward,
    providers=ATTENTION_PROVIDERS,
    autotune_kernels=flash_attention_autotune_kernels(),
    nsight_size=attention_element_count(128),
    nsight_autotune_kernels=flash_attention_autotune_kernels,
)

attention_summary = to_summary_dataframe(attention_rows)
attention_failures = attention_summary[attention_summary["status"] != "ok"]
if not attention_failures.empty:
    display(attention_failures[["provider", "size", "status", "error"]])


### Correctness contracts

These checks validate dropout replay, every compiled fp16 bucket, non-contiguous layouts, CUDA streams, and capability boundaries.


In [ ]:
def check_triton_dropout_correctness():
    release_cache()
    try:
        batch, heads, seq, head_dim = 1, 2, 64, 32
        shape = (batch, heads, seq, head_dim)
        q, k, v = (torch.randn(shape, device=DEVICE, dtype=torch.float32) for _ in range(3))
        if not is_triton_flash_attention_supported(
            q, k, v, dropout_p=ATTENTION_DROPOUT_RATE
        ):
            return {"status": "unsupported"}

        seed = torch.tensor(1234567, device=DEVICE, dtype=torch.int32)
        grad_out = torch.randn_like(q)
        out, lse, _, softmax_scale = flash_attention_forward(
            q,
            k,
            v,
            is_causal=ATTENTION_CAUSAL,
            dropout_p=ATTENTION_DROPOUT_RATE,
            dropout_seed=seed,
        )
        keep = flash_attention_dropout_mask(
            batch,
            heads,
            seq,
            seq,
            device=DEVICE,
            dropout_p=ATTENTION_DROPOUT_RATE,
            dropout_seed=seed,
        )

        q_ref, k_ref, v_ref = (tensor.detach().clone().requires_grad_(True) for tensor in (q, k, v))
        scores = q_ref @ k_ref.transpose(-1, -2) / math.sqrt(head_dim)
        if ATTENTION_CAUSAL:
            positions = torch.arange(seq, device=DEVICE)
            scores = scores.masked_fill(positions[:, None] < positions[None, :], float("-inf"))
        probabilities = torch.softmax(scores, dim=-1)
        reference = (probabilities * keep / (1.0 - ATTENTION_DROPOUT_RATE)) @ v_ref
        reference.backward(grad_out)

        dq, dk, dv = flash_attention_backward(
            grad_out,
            q,
            k,
            v,
            out,
            lse,
            seed,
            is_causal=ATTENTION_CAUSAL,
            dropout_p=ATTENTION_DROPOUT_RATE,
            softmax_scale=softmax_scale,
        )
        torch.cuda.synchronize()

        pairs = {
            "fwd": (out, reference),
            "dq": (dq, q_ref.grad),
            "dk": (dk, k_ref.grad),
            "dv": (dv, v_ref.grad),
        }
        stats = {"status": "ok", "keep_ratio": keep.float().mean().item()}
        for name, (actual, expected) in pairs.items():
            actual, expected = actual.float(), expected.float()
            stats[f"{name}_max_abs"] = (actual - expected).abs().max().item()
            stats[f"{name}_correct"] = torch.allclose(actual, expected, atol=3e-2, rtol=3e-2)
        return stats
    finally:
        release_cache()


In [ ]:
def check_cuda_fp16_correctness():
    """Validate CUDA fp16 buckets and masked tails against an fp32 reference."""

    # Keep the debug problem deliberately small: the helper materializes the S x S
    # dropout matrix solely so PyTorch can replay the CUDA kernel's exact random mask.
    batch, heads, seq = 1, 2, 37
    # Exact bucket boundaries exercise the primary specializations. Values
    # between them exercise uneven-K masked loads/stores in the next bucket.
    head_dims = (32, 40, 64, 80, 96, 128, 160, 192, 200, 256)
    rows = []
    release_cache()
    try:
        for head_dim in head_dims:
            for is_causal in (False, True):
                for dropout_p in (0.0, ATTENTION_DROPOUT_RATE):
                    shape = (batch, heads, seq, head_dim)
                    q, k, v = (
                        torch.randn(shape, device=DEVICE, dtype=torch.float16)
                        for _ in range(3)
                    )
                    row = {
                        "dtype": "fp16",
                        "head_dim": head_dim,
                        "causal": is_causal,
                        "dropout_p": dropout_p,
                    }

                    # The support predicate distinguishes omitted build variants and
                    # hardware limits such as sm86 D256 dropout backward shared memory.
                    if not is_cuda_flash_attention_supported(
                        q, k, v, dropout_p=dropout_p
                    ):
                        row["status"] = "unsupported"
                        rows.append(row)
                        continue

                    grad_out = torch.randn_like(q)

                    # Run the production autograd path first. Resetting the CUDA seed
                    # lets the debug helper reproduce this call's exact Philox mask.
                    seed = 20260713 + head_dim * 10 + int(is_causal)
                    torch.cuda.manual_seed(seed)
                    rng_before = torch.cuda.get_rng_state(q.device)
                    q_cuda, k_cuda, v_cuda = (
                        tensor.detach().clone().requires_grad_(True) for tensor in (q, k, v)
                    )
                    out = get_ops_backend("cuda").attention(
                        q_cuda,
                        k_cuda,
                        v_cuda,
                        is_causal=is_causal,
                        dropout_p=dropout_p,
                    )
                    out.backward(grad_out)
                    rng_after = torch.cuda.get_rng_state(q.device)
                    rng_advanced = not torch.equal(rng_before, rng_after)
                    row["rng_behavior_correct"] = bool(
                        rng_advanced if dropout_p else not rng_advanced
                    )
                    cuda_grads = (q_cuda.grad, k_cuda.grad, v_cuda.grad)

                    # For dropout, read the keep mask produced by the vendored CUDA
                    # kernel. For no-dropout, an all-true mask gives the same formula.
                    if dropout_p:
                        torch.cuda.manual_seed(seed)
                        debug_out, keep = cuda_flash_attention_dropout_mask(
                            q, k, v,
                            is_causal=is_causal,
                            dropout_p=dropout_p,
                        )
                        row["debug_output_equal"] = bool(torch.equal(out, debug_out))
                        row["keep_ratio"] = keep.float().mean().item()
                    else:
                        keep = torch.ones(
                            (batch, heads, seq, seq), device=DEVICE, dtype=torch.bool
                        )
                        row["debug_output_equal"] = True
                        row["keep_ratio"] = 1.0

                    # Compute attention in fp32 so the reference does not silently
                    # select another fused attention implementation in PyTorch SDPA.
                    q_ref, k_ref, v_ref = (
                        tensor.detach().float().requires_grad_(True) for tensor in (q, k, v)
                    )
                    scores = q_ref @ k_ref.transpose(-1, -2) / math.sqrt(head_dim)
                    if is_causal:
                        causal_mask = torch.ones(
                            (seq, seq), device=DEVICE, dtype=torch.bool
                        ).triu(1)
                        scores = scores.masked_fill(causal_mask, float("-inf"))
                    probabilities = torch.softmax(scores, dim=-1)
                    reference = (
                        probabilities * keep / (1.0 - dropout_p)
                    ) @ v_ref
                    # A scalar sum produces an expanded stride-0 dout in
                    # autograd. Save its fp32 gradients before consuming the
                    # reference graph with the normal random grad_out check.
                    reference_sum_grads = torch.autograd.grad(
                        reference.sum(), (q_ref, k_ref, v_ref), retain_graph=True
                    )
                    reference.backward(grad_out.float())

                    # Report each derivative separately; one aggregate boolean would
                    # hide which side of backward failed when a branch regresses.
                    pairs = {
                        "fwd": (out, reference),
                        "dq": (cuda_grads[0], q_ref.grad),
                        "dk": (cuda_grads[1], k_ref.grad),
                        "dv": (cuda_grads[2], v_ref.grad),
                    }
                    row["status"] = "ok"
                    for name, (actual, expected) in pairs.items():
                        actual = actual.float()
                        expected = expected.float()
                        row[f"{name}_max_abs"] = (actual - expected).abs().max().item()
                        row[f"{name}_correct"] = bool(
                            torch.allclose(actual, expected, atol=3e-2, rtol=3e-2)
                        )

                    # Exercise the adapter's non-contiguous dout conversion for
                    # every branch, using the same Philox mask for dropout.
                    torch.cuda.manual_seed(seed)
                    q_sum, k_sum, v_sum = (
                        tensor.detach().clone().requires_grad_(True)
                        for tensor in (q, k, v)
                    )
                    sum_output = get_ops_backend("cuda").attention(
                        q_sum,
                        k_sum,
                        v_sum,
                        is_causal=is_causal,
                        dropout_p=dropout_p,
                    )
                    sum_output.sum().backward()
                    row["expanded_dout_correct"] = all(
                        torch.allclose(actual.float(), expected, atol=3e-2, rtol=3e-2)
                        for actual, expected in zip(
                            (q_sum.grad, k_sum.grad, v_sum.grad), reference_sum_grads
                        )
                    )
                    row["all_correct"] = (
                        row["debug_output_equal"]
                        and row["rng_behavior_correct"]
                        and row["expanded_dout_correct"]
                        and all(
                            row[f"{name}_correct"]
                            for name in ("fwd", "dq", "dk", "dv")
                        )
                    )
                    rows.append(row)
        torch.cuda.synchronize()
        return rows
    finally:
        release_cache()


In [ ]:
def check_cuda_layout_and_stream():
    """Validate outer strides and launches on a non-default CUDA stream."""

    batch, heads, seq, head_dim = 1, 2, 37, 128
    rows = []
    release_cache()
    try:
        for is_causal in (False, True):
            for dropout_p in (0.0, ATTENTION_DROPOUT_RATE):
                # Start in B,S,H,D and transpose without copying. The resulting
                # B,H,S,D tensors keep stride(-1)==1 but are not contiguous.
                physical_shape = (batch, seq, heads, head_dim)
                q, k, v = (
                    torch.randn(
                        physical_shape, device=DEVICE, dtype=torch.float16
                    ).transpose(1, 2)
                    for _ in range(3)
                )
                row = {
                    "dtype": "fp16",
                    "head_dim": head_dim,
                    "causal": is_causal,
                    "dropout_p": dropout_p,
                    "input_stride": tuple(q.stride()),
                    "outer_noncontiguous": (
                        not q.is_contiguous() and q.stride(-1) == 1
                    ),
                }
                if not is_cuda_flash_attention_supported(
                    q, k, v, dropout_p=dropout_p
                ):
                    row["status"] = "unsupported"
                    rows.append(row)
                    continue

                grad_out = torch.randn_like(q)
                seed = 20260713 + int(is_causal)
                stream = torch.cuda.Stream(device=q.device)
                torch.cuda.synchronize(q.device)

                # Keep forward, backward, and debug-mask extraction on the same
                # explicit stream. Synchronize that stream only before CPU checks.
                with torch.cuda.stream(stream):
                    torch.cuda.manual_seed(seed)
                    q_cuda, k_cuda, v_cuda = (
                        tensor.detach().clone().requires_grad_(True)
                        for tensor in (q, k, v)
                    )
                    out = get_ops_backend("cuda").attention(
                        q_cuda,
                        k_cuda,
                        v_cuda,
                        is_causal=is_causal,
                        dropout_p=dropout_p,
                    )
                    out.backward(grad_out)
                    cuda_grads = (q_cuda.grad, k_cuda.grad, v_cuda.grad)
                    if dropout_p:
                        torch.cuda.manual_seed(seed)
                        debug_out, keep = cuda_flash_attention_dropout_mask(
                            q,
                            k,
                            v,
                            is_causal=is_causal,
                            dropout_p=dropout_p,
                        )
                    else:
                        debug_out = out
                        keep = torch.ones(
                            (batch, heads, seq, seq),
                            device=DEVICE,
                            dtype=torch.bool,
                        )
                    completion = torch.cuda.Event()
                    completion.record(stream)
                completion.synchronize()

                # Build the reference after stream completion. It consumes the
                # exact CUDA keep mask but uses explicit fp32 matmul/softmax.
                q_ref, k_ref, v_ref = (
                    tensor.detach().float().requires_grad_(True)
                    for tensor in (q, k, v)
                )
                scores = q_ref @ k_ref.transpose(-1, -2) / math.sqrt(head_dim)
                if is_causal:
                    causal_mask = torch.ones(
                        (seq, seq), device=DEVICE, dtype=torch.bool
                    ).triu(1)
                    scores = scores.masked_fill(causal_mask, float("-inf"))
                reference = (
                    torch.softmax(scores, dim=-1)
                    * keep
                    / (1.0 - dropout_p)
                ) @ v_ref
                reference.backward(grad_out.float())

                pairs = {
                    "fwd": (out, reference),
                    "dq": (cuda_grads[0], q_ref.grad),
                    "dk": (cuda_grads[1], k_ref.grad),
                    "dv": (cuda_grads[2], v_ref.grad),
                }
                row.update(
                    status="ok",
                    stream_id=stream.cuda_stream,
                    kernel_input_stride=tuple(q_cuda.stride()),
                    kernel_outer_noncontiguous=(
                        not q_cuda.is_contiguous() and q_cuda.stride(-1) == 1
                    ),
                    output_stride=tuple(out.stride()),
                    debug_output_equal=bool(torch.equal(out, debug_out)),
                )
                for name, (actual, expected) in pairs.items():
                    row[f"{name}_correct"] = bool(
                        torch.allclose(
                            actual.float(), expected, atol=3e-2, rtol=3e-2
                        )
                    )
                row["all_correct"] = (
                    row["outer_noncontiguous"]
                    and row["kernel_outer_noncontiguous"]
                    and row["debug_output_equal"]
                    and all(
                        row[f"{name}_correct"]
                        for name in ("fwd", "dq", "dk", "dv")
                    )
                )
                rows.append(row)
        return rows
    finally:
        release_cache()


In [ ]:
def check_cuda_capability_contract():
    """Check no-load CUDA capability boundaries for the active build."""

    config = get_cuda_build_config()
    capability = torch.cuda.get_device_capability(DEVICE)
    batch, heads, seq = 1, 2, 7

    # Build one valid control and derive the expected result from the active
    # profile. A larger linked bucket can serve a smaller multiple-of-8 D.
    def build_has(head_dim):
        return (
            "fp16" in config.dtypes
            and any(head_dim <= bucket for bucket in config.head_dims)
            and capability[0] >= 8
        )

    def tensors(shape, *, dtype=torch.float16):
        return tuple(
            torch.randn(shape, device=DEVICE, dtype=dtype) for _ in range(3)
        )

    valid = tensors((batch, heads, seq, 128))
    d200 = tensors((batch, heads, seq, 200))
    # Reproduce the public 144-KiB hardware contract independently. Older
    # PyTorch versions omit the opt-in field, so use the same audited CUDA
    # architecture facts while keeping the threshold comparison visible.
    properties = torch.cuda.get_device_properties(DEVICE)
    optin_smem = next((
        int(getattr(properties, name))
        for name in (
            "shared_memory_per_block_optin",
            "max_shared_memory_per_block_optin",
        )
        if getattr(properties, name, None) is not None
    ), {
        (8, 0): 163 * 1024, (8, 6): 99 * 1024,
        (8, 7): 163 * 1024, (8, 9): 99 * 1024,
        (9, 0): 227 * 1024,
    }.get(capability))
    d200_dropout_expected = (
        build_has(200)
        and optin_smem is not None
        and optin_smem >= 144 * 1024
    )

    # Every invalid case must be rejected before load_cuda_extension(). The
    # two controls track profile and hardware-dependent support explicitly.
    cases = [
        ("valid_d128", valid, 0.0, build_has(128)),
        ("empty_batch", tensors((0, heads, seq, 128)), 0.0, False),
        ("empty_heads", tensors((batch, 0, seq, 128)), 0.0, False),
        ("empty_sequence", tensors((batch, heads, 0, 128)), 0.0, False),
        ("fp32_dtype", tensors((batch, heads, seq, 128), dtype=torch.float32), 0.0, False),
        ("head_dim_not_multiple_of_8", tensors((batch, heads, seq, 130)), 0.0, False),
        ("head_dim_over_256", tensors((batch, heads, seq, 264)), 0.0, False),
        ("dropout_out_of_range", valid, 1.0, False),
        ("dropout_rounds_to_float32_one", valid, 1.0 - 1e-12, False),
        ("dropout_underflows_to_float32_zero", valid, 1e-50, build_has(128)),
        ("d200_dropout_control", d200, ATTENTION_DROPOUT_RATE, d200_dropout_expected),
    ]

    # Construct shape and stride violations separately so only one contract
    # changes in each row. Slicing every other D keeps shape 128, stride 2.
    mismatch = list(valid)
    mismatch[1] = tensors((batch, heads, seq + 1, 128))[1]
    cases.append(("shape_mismatch", tuple(mismatch), 0.0, False))
    noncontiguous = tuple(
        tensor[..., ::2] for tensor in tensors((batch, heads, seq, 256))
    )
    cases.append(("head_dim_stride_2", noncontiguous, 0.0, False))

    rows = []
    for name, (q, k, v), dropout_p, expected in cases:
        actual = is_cuda_flash_attention_supported(
            q, k, v, dropout_p=dropout_p
        )
        rows.append(
            {
                "case": name,
                "shape": tuple(q.shape),
                "last_stride": q.stride(-1),
                "dropout_p": dropout_p,
                "supported": actual,
                "expected": expected,
                "correct": actual == expected,
            }
        )
    return rows


### Dropout validation and sequence-length sweep

Supported branches must pass correctness before timings are reported.


In [ ]:
triton_dropout_check = check_triton_dropout_correctness()
cuda_fp16_check_rows = check_cuda_fp16_correctness()
cuda_layout_stream_check_rows = check_cuda_layout_and_stream()
cuda_capability_check_rows = check_cuda_capability_contract()

cache_benchmark_rows(
    "attention_dropout_triton_correctness",
    [triton_dropout_check],
    dataset_type="correctness",
)
cache_benchmark_rows(
    "attention_cuda_fp16_correctness",
    cuda_fp16_check_rows,
    dataset_type="correctness",
)
cache_benchmark_rows(
    "attention_cuda_layout_stream_correctness",
    cuda_layout_stream_check_rows,
    dataset_type="correctness",
)
cache_benchmark_rows(
    "attention_cuda_capability_contract",
    cuda_capability_check_rows,
    dataset_type="correctness",
)

cuda_fp16_check_frame = pd.DataFrame(cuda_fp16_check_rows)
display(cuda_fp16_check_frame)
display(pd.DataFrame(cuda_layout_stream_check_rows))
display(pd.DataFrame(cuda_capability_check_rows))

cuda_fp16_failures = cuda_fp16_check_frame[
    (cuda_fp16_check_frame["status"] == "ok")
    & (cuda_fp16_check_frame["all_correct"] != True)
]
layout_stream_failures = [
    row for row in cuda_layout_stream_check_rows if not row["all_correct"]
]
capability_failures = [
    row for row in cuda_capability_check_rows if not row["correct"]
]

print(
    "CUDA fp16 correctness: "
    f"{(cuda_fp16_check_frame['status'] == 'ok').sum()} supported, "
    f"{(cuda_fp16_check_frame['status'] == 'unsupported').sum()} unsupported, "
    f"{len(cuda_fp16_failures)} failed"
)
if not cuda_fp16_failures.empty:
    display(cuda_fp16_failures)

# Supported branches are executable tests: invalid timings must never enter reports.
assert cuda_fp16_failures.empty, (
    f"CUDA fp16 numerical failures: {len(cuda_fp16_failures)}"
)
assert not layout_stream_failures, (
    f"CUDA layout/stream failures: {layout_stream_failures}"
)
assert not capability_failures, (
    f"CUDA capability failures: {capability_failures}"
)

cuda_dropout_rows = [
    row
    for row in cuda_fp16_check_rows
    if row["status"] == "ok" and row["dropout_p"] == ATTENTION_DROPOUT_RATE
]
dropout_correctness = {
    "torch": {"fwd_correct": True, "bwd_correct": True},
    "triton": {
        "fwd_correct": triton_dropout_check.get("fwd_correct", False),
        "bwd_correct": all(
            triton_dropout_check.get(f"{name}_correct", False)
            for name in ("dq", "dk", "dv")
        ),
    },
    "cuda": {
        "fwd_correct": bool(cuda_dropout_rows) and all(
            row["fwd_correct"]
            and row["debug_output_equal"]
            and row["rng_behavior_correct"]
            for row in cuda_dropout_rows
        ),
        "bwd_correct": bool(cuda_dropout_rows) and all(
            row[name]
            for row in cuda_dropout_rows
            for name in ("dq_correct", "dk_correct", "dv_correct")
        ),
    },
}

display(pd.DataFrame([{
    "status": triton_dropout_check.get("status"),
    "dropout_p": ATTENTION_DROPOUT_RATE,
    "keep_ratio": triton_dropout_check.get("keep_ratio"),
}]))
display(pd.DataFrame([
    {
        "check": name,
        "correct": triton_dropout_check.get(f"{name}_correct"),
        "max_abs": triton_dropout_check.get(f"{name}_max_abs"),
    }
    for name in ("fwd", "dq", "dk", "dv")
]))

attention_dropout_forward = make_attention_forward(
    is_causal=ATTENTION_CAUSAL,
    dropout_p=ATTENTION_DROPOUT_RATE,
)
register_nsight_kernel(
    "attention_dropout",
    make_attention_case,
    attention_dropout_forward,
    default_size=attention_element_count(128),
    autotune_kernels=flash_attention_autotune_kernels,
)
report_pytorch_sdpa_backends(attention_sizes, dropout_p=ATTENTION_DROPOUT_RATE)

attention_dropout_rows = [
    row
    for size in attention_sizes
    for row in benchmark_attention_case(
        kernel="attention_dropout",
        size=size,
        make_case=make_attention_case,
        forward=attention_dropout_forward,
        correctness=dropout_correctness,
    )
]
cache_benchmark_rows(
    "attention_dropout",
    attention_dropout_rows,
    providers=ATTENTION_PROVIDERS,
    sequence_lengths=ATTENTION_SEQUENCE_LENGTHS,
    dropout_p=ATTENTION_DROPOUT_RATE,
)
display_triton_config_summary(attention_dropout_rows)
display(to_summary_dataframe(attention_dropout_rows))
_ = plot_kernel_grid(
    attention_dropout_rows,
    metrics=METRICS,
    save_path=FIG_DIR / "attention_dropout_summary.png",
)


### CUDA fp16 specialization matrix

This fixes sequence length at 1024 and traverses every compiled head-dimension, causal, and dropout branch.


In [ ]:
# Fixed-S matrix: every compiled fp16 head-dimension, causal, and dropout branch.
CUDA_MATRIX_BATCH_SIZE = 1
CUDA_MATRIX_NUM_HEADS = 4
CUDA_MATRIX_SEQUENCE_LENGTH = 512
CUDA_MATRIX_HEAD_DIMS = (32, 64, 96, 128, 192, 256)
CUDA_MATRIX_CAUSAL_OPTIONS = (False, True)
CUDA_MATRIX_DROPOUT_RATES = (0.0, ATTENTION_DROPOUT_RATE)


def make_cuda_matrix_case(size, *, head_dim):
    return make_attention_case(
        size,
        batch_size=CUDA_MATRIX_BATCH_SIZE,
        num_heads=CUDA_MATRIX_NUM_HEADS,
        head_dim=head_dim,
        dtype=torch.float16,
    )


def run_cuda_fp16_performance_matrix():
    correctness_index = {
        (row["head_dim"], row["causal"], row["dropout_p"]): row
        for row in cuda_fp16_check_rows
    }
    rows = []

    branches = product(
        CUDA_MATRIX_HEAD_DIMS,
        CUDA_MATRIX_CAUSAL_OPTIONS,
        CUDA_MATRIX_DROPOUT_RATES,
    )
    for head_dim, is_causal, dropout_p in branches:
        size = attention_element_count(
            CUDA_MATRIX_SEQUENCE_LENGTH,
            batch_size=CUDA_MATRIX_BATCH_SIZE,
            num_heads=CUDA_MATRIX_NUM_HEADS,
            head_dim=head_dim,
        )
        make_case = partial(make_cuda_matrix_case, head_dim=head_dim)
        forward = make_attention_forward(
            is_causal=is_causal,
            dropout_p=dropout_p,
        )

        probe = make_case(size)
        try:
            supported = attention_provider_support(
                probe.tensors,
                dropout_p=dropout_p,
            )
        finally:
            probe.tensors.clear()
            release_cache()

        branch_check = correctness_index[(head_dim, is_causal, dropout_p)]
        correctness = {
            "cuda": {
                "fwd_correct": branch_check.get("fwd_correct"),
                "bwd_correct": all(
                    branch_check.get(name, False)
                    for name in ("dq_correct", "dk_correct", "dv_correct")
                ),
            }
        }
        rows.extend(
            benchmark_attention_case(
                kernel="attention_fp16_matrix",
                size=size,
                make_case=make_case,
                forward=forward,
                metadata={
                    "dtype": "fp16",
                    "head_dim": head_dim,
                    "sequence": CUDA_MATRIX_SEQUENCE_LENGTH,
                    "causal": is_causal,
                    "dropout_p": dropout_p,
                },
                correctness=correctness,
                supported=supported,
            )
        )
    return rows


cuda_fp16_performance_rows = run_cuda_fp16_performance_matrix()
cache_benchmark_rows(
    "attention_fp16_matrix",
    cuda_fp16_performance_rows,
    providers=ATTENTION_PROVIDERS,
    sequence_length=CUDA_MATRIX_SEQUENCE_LENGTH,
    head_dims=CUDA_MATRIX_HEAD_DIMS,
    causal_options=CUDA_MATRIX_CAUSAL_OPTIONS,
    dropout_rates=CUDA_MATRIX_DROPOUT_RATES,
)
display_triton_config_summary(cuda_fp16_performance_rows)
cuda_fp16_performance_frame = pd.DataFrame(cuda_fp16_performance_rows)
cuda_fp16_performance_columns = (
    "dtype", "head_dim", "sequence", "causal", "dropout_p",
    "provider", "status", "fwd_correct", "bwd_correct",
    "fwd_p50_ms", "fwd_p95_ms", "fwd_peak_mem_mb", "fwd_speedup",
    "bwd_p50_ms", "bwd_p95_ms", "bwd_peak_mem_mb", "bwd_speedup",
    "error",
)
display(cuda_fp16_performance_frame[[
    column
    for column in cuda_fp16_performance_columns
    if column in cuda_fp16_performance_frame.columns
]])


### Nsight Compute

The call below profiles one backward pass at sequence length 128 in a clean worker process. It writes both `.ncu-rep` and `.csv` reports under the active GPU's `tests/benchmark_results/<gpu>/nsight` directory.


In [ ]:
try:
    attention_nsight = profile_and_cache_kernel(
        "attention",
        mode="bwd",
        size=attention_element_count(128),
        sections=("LaunchStats",),
        timeout_s=120,
    )
except PermissionError as error:
    print(error)


### Nsight Compute (standalone)

This block uses the same registration path as `run_kernel`, but it only registers a small FlashAttention case for Nsight. It does not run the benchmark sweep.


In [ ]:
import torch

from minitrain.kernels.triton.flash_attention import flash_attention_autotune_kernels
from minitrain.model.ops import get_ops_backend
from operator_bench_utils import BenchCase

STANDALONE_ATTENTION_BATCH = 1
STANDALONE_ATTENTION_HEADS = 4
STANDALONE_ATTENTION_HEAD_DIM = 128
STANDALONE_ATTENTION_CAUSAL = True
STANDALONE_ATTENTION_DROPOUT_P = 0.0
STANDALONE_ATTENTION_DEVICE = torch.device("cuda")
STANDALONE_ATTENTION_DTYPE = torch.float16


def standalone_attention_size(seq):
    return (
        3
        * STANDALONE_ATTENTION_BATCH
        * STANDALONE_ATTENTION_HEADS
        * seq
        * STANDALONE_ATTENTION_HEAD_DIM
    )


def make_standalone_attention_case(size):
    seq = size // (
        3
        * STANDALONE_ATTENTION_BATCH
        * STANDALONE_ATTENTION_HEADS
        * STANDALONE_ATTENTION_HEAD_DIM
    )
    shape = (
        STANDALONE_ATTENTION_BATCH,
        STANDALONE_ATTENTION_HEADS,
        seq,
        STANDALONE_ATTENTION_HEAD_DIM,
    )
    return BenchCase(
        tensors={
            name: torch.randn(
                shape,
                device=STANDALONE_ATTENTION_DEVICE,
                dtype=STANDALONE_ATTENTION_DTYPE,
            )
            for name in ("q", "k", "v")
        },
        grad_names=("q", "k", "v"),
    )


def standalone_attention_forward(provider, tensors):
    return get_ops_backend(provider).attention(
        tensors["q"],
        tensors["k"],
        tensors["v"],
        is_causal=STANDALONE_ATTENTION_CAUSAL,
        dropout_p=STANDALONE_ATTENTION_DROPOUT_P,
    )


standalone_attention_sizes = (standalone_attention_size(8*1024),)
register_kernel(
    "standalone_attention",
    standalone_attention_sizes,
    "Q + K + V elements",
    make_standalone_attention_case,
    standalone_attention_forward,
    nsight_size=standalone_attention_size(8*1024),
    nsight_autotune_kernels=flash_attention_autotune_kernels,
)

# Example: uncomment one line when you want to generate a small report.
# standalone_attention_report = profile_and_cache_kernel("standalone_attention", mode="fwd", timeout_s=120)
standalone_attention_bwd_report = profile_and_cache_kernel(
    "standalone_attention",
    mode="bwd",
    timeout_s=300,
    launch_count=32,
    kernel_filter="_flash_bwd",
    capture_kernel_filter=".*_flash_bwd.*",
    capture_mode="cuda_profiler",
)


### Some debug tools

In [ ]:
# find all unique kernel names in the report
import pandas as pd
pd.DataFrame(standalone_attention_bwd_report["summary_all"])[["KernelName"]].drop_duplicates()

In [ ]:
# check the GPU properties and capabilities
def get_prop(props, *names):
    for name in names:
        if hasattr(props, name):
            return getattr(props, name)
    return None

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda)
print("device count:", torch.cuda.device_count())

print(torch.cuda.get_device_capability(0))

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"\nGPU {i}: {props}")
    # print(f"\nGPU {i}: {props.name}")
    # print("compute capability:", props.major, props.minor)
    # print("total memory GB:", props.total_memory / 1024**3)
    # print("multi processor count:",
    # props.multi_processor_count)
    # print("warp size:", get_prop(props, "warp_size"))
    # print("max threads per block:", get_prop(props,
    # "max_threads_per_block"))
    # print("max threads per multiprocessor:", get_prop(props,
    # "max_threads_per_multiprocessor"))

    # print("registers per block:", get_prop(props,
    # "regs_per_block"))
    # print("registers per multiprocessor:", get_prop(props,
    # "regs_per_multiprocessor"))

    # smem_block = get_prop(
    #     props,
    #     "shared_memory_per_block",
    #     "shared_memory_per_block_optin",
    # )
    # smem_sm = get_prop(
    #     props,
    #     "shared_memory_per_multiprocessor",
    #     "shared_memory_per_sm",
    # )

    # print("shared memory per block KB:", None if smem_block
    # is None else smem_block / 1024)
    # print("shared memory per multiprocessor KB:", None if
    # smem_sm is None else smem_sm / 1024)

In [ ]:
# txt for the report
from pathlib import Path

report_path = Path(standalone_attention_bwd_report["report"])
details_path = report_path.with_suffix(".details.txt")

print(report_path)
print(report_path.exists())

# Show the complete Nsight Compute details page in the notebook output.
!ncu --import "{report_path}" --page details

# Save the same complete details page next to the .ncu-rep report.
!ncu --import "{report_path}" --page details > "{details_path}"

print(details_path)
